# Notebook 4 — Optimisation, seuil métier et feature importance

Ce notebook :
1. Optimise les hyperparamètres de LightGBM avec **Optuna** (objectif = coût métier)
2. Trouve le **seuil de décision optimal** selon le coût métier
3. Analyse la **feature importance globale** (LightGBM built-in)
4. Analyse la **feature importance locale** avec SHAP (beeswarm + waterfall)

In [2]:
import mlflow
import optuna
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, recall_score
from lightgbm import LGBMClassifier
import sys
sys.path.insert(0, '..')
from src.modelization import cout_metier, trouver_seuil_optimal
from src.preprocessing import build_preprocessor
from src.visualizer import create_lineplot, create_barh
from mlflow.sklearn import log_model as mlflow_log_model

RAMDOM_STATE = 42
# Réduire le verbosity d'Optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("credit-scoring-optimization")

<Experiment: artifact_location='/home/rapha/ai-engineer/credit-scoring-mlops/notebooks/mlruns/2', creation_time=1773772076389, experiment_id='2', last_update_time=1773772076389, lifecycle_stage='active', name='credit-scoring-optimization', tags={}, workspace='default'>

## Chargement des données

In [ ]:
df_train = pd.read_csv("../data/processed/train_processed.csv")

X = df_train.drop(columns=["TARGET", "SK_ID_CURR"])
y = df_train["TARGET"]

# Split train/validation — fait EN PREMIER avant toute optimisation
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RAMDOM_STATE
)

print(f"Train : {X_train.shape[0]} lignes | Validation : {X_val.shape[0]} lignes")
print(f"Distribution cible (train) :\n{y_train.value_counts(normalize=True).round(3)}")

Train : 246008 lignes | Validation : 61503 lignes
Distribution cible (train) :
TARGET
0    0.919
1    0.081
Name: proportion, dtype: float64


In [3]:
# Identification des types de colonnes
num_cols = X_train.select_dtypes(include='number').columns.tolist()
cat_cols = X_train.select_dtypes(include='object').columns.tolist()

print(f'Colonnes numériques   : {len(num_cols)}')
print(f'Colonnes catégorielles : {len(cat_cols)}')

Colonnes numériques   : 98
Colonnes catégorielles : 13


## Optimisation des hyperparamètres avec Optuna

Optuna explore l'espace des hyperparamètres de façon intelligente (optimisation bayésienne via TPE)
plutôt qu'une grille exhaustive.

**Objectif : maximiser l'AUC** par validation croisée à 3 folds sur `X_train` uniquement.

> **Note :** On optimise sur l'AUC (et non le coût métier) car l'AUC est calculable directement
> dans la CV sans chercher un seuil à chaque fold. Le seuil optimal est déterminé **une seule fois**
> après, sur `X_val`.
> Avantage par rapport à évaluer directement sur `X_val` : `X_val` n'est jamais vu pendant
> l'optimisation, ce qui évite le data leakage. En contrepartie, on réduit le nombre de trials
> (15 au lieu de 30) pour compenser le coût de la CV (15 × 3 folds = 45 entraînements).

In [ ]:
import warnings
warnings.filterwarnings('ignore', message='X does not have valid feature names')

def objective(trial):
    """Fonction objectif Optuna — maximise l'AUC par CV 3 folds sur X_train.

    X_val n'est jamais utilisé ici, ce qui évite le data leakage.
    Optuna minimise → on retourne l'opposé de l'AUC.
    """
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 600, step=100),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 20, 100),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 60),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-8, 1.0, log=True),
    }

    pipeline = Pipeline([
        ('preprocessor', build_preprocessor(num_cols, cat_cols)),
        ('clf', LGBMClassifier(
            **params,
            class_weight='balanced',
            random_state=RAMDOM_STATE,
            n_jobs=-1,
            verbose=-1,
        ))
    ])

    # CV 3 folds sur X_train uniquement — X_val n'est jamais vu pendant l'optimisation
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RAMDOM_STATE)
    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=1)
    return -scores.mean()  # Optuna minimise → on retourne l'opposé de l'AUC


study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=RAMDOM_STATE)
)
study.optimize(objective, n_trials=15, show_progress_bar=True)

best_auc_cv = -study.best_value
print(f"\nMeilleure AUC (CV) : {best_auc_cv:.4f}")
print(f"Meilleurs paramètres : {study.best_params}")

In [ ]:
# Courbe de convergence d'Optuna — AUC par essai
trials_df = study.trials_dataframe()
auc_values = -trials_df['value']  # reconversion en AUC positif (Optuna stocke -AUC)

fig, ax = plt.subplots(figsize=(10, 4))
create_lineplot(
    ax,
    x=trials_df['number'], y=auc_values,
    marker='o', alpha=0.6,
    label='AUC par essai',
    hline=best_auc_cv, hline_label=f'Meilleur AUC : {best_auc_cv:.4f}',
    xlabel='Essai Optuna',
    ylabel='AUC (validation croisée 3 folds)',
    title='Convergence Optuna — AUC moyen par essai',
)
plt.tight_layout()
plt.savefig('../data/processed/optuna_convergence.png', dpi=150, bbox_inches='tight')
plt.show()

## Entraînement du modèle final optimisé

In [ ]:
# Entraînement avec les meilleurs hyperparamètres trouvés par Optuna
best_pipeline = Pipeline([
    ('preprocessor', build_preprocessor(num_cols, cat_cols)),
    ('clf', LGBMClassifier(
        **study.best_params,
        class_weight='balanced',
        random_state=RAMDOM_STATE,
        n_jobs=-1,
        verbose=-1,
    ))
])

best_pipeline.fit(X_train, y_train)
y_pred_proba = best_pipeline.predict_proba(X_val)[:, 1]

print("Modèle final entraîné.")

Modèle final entraîné.


/home/rapha/ai-engineer/credit-scoring-mlops/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
# Recherche du seuil optimal et tracé de la courbe coût vs seuil
seuils = np.arange(0.05, 0.95, 0.01)
couts  = [cout_metier(y_val, y_pred_proba, seuil=s) for s in seuils]

seuil_opt, cout_opt = trouver_seuil_optimal(y_val, y_pred_proba)

fig, ax = plt.subplots(figsize=(10, 5))
create_lineplot(
    ax,
    x=seuils, y=couts,
    vline=seuil_opt, vline_label=f'Seuil optimal = {seuil_opt}',
    xlabel='Seuil de décision',
    ylabel='Coût métier (FN×10 + FP×1)',
    title='Coût métier en fonction du seuil de décision',
)
plt.tight_layout()
plt.savefig('../data/processed/courbe_seuil.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Seuil optimal : {seuil_opt} | Coût métier minimal : {cout_opt:,}")

In [ ]:
# Métriques finales
y_pred_final = (y_pred_proba >= seuil_opt).astype(int)
roc_auc_final    = roc_auc_score(y_val, y_pred_proba)
recall_final = recall_score(y_val, y_pred_final)

with mlflow.start_run(run_name='lgbm_optimized') as run:
    # Paramètres
    mlflow.log_params(study.best_params)
    mlflow.log_param('n_trials_optuna', 15)
    mlflow.log_param('cv_folds_optuna', 3)
    mlflow.log_metric('seuil_optimal', seuil_opt)

    # Métriques
    mlflow.log_metric('roc_auc', round(roc_auc_final, 4))
    mlflow.log_metric('roc_auc_cv_optuna', round(best_auc_cv,   4))
    mlflow.log_metric('cout_metier', cout_opt)
    mlflow.log_metric('recall_classe_1', round(recall_final,  4))

    # Artefacts
    mlflow.log_artifact('../data/processed/courbe_seuil.png')
    mlflow.log_artifact('../data/processed/optuna_convergence.png')

    # Modèle
    mlflow_log_model(best_pipeline, name='model')
    mlflow.set_tag('dataset', 'full')
    mlflow.set_tag('etape', 'optimisation')

    run_id = run.info.run_id

print(f"AUC (val)       : {roc_auc_final:.4f}")
print(f"AUC (CV Optuna) : {best_auc_cv:.4f}")
print(f"Recall classe 1 : {recall_final:.4f}")
print(f"Coût métier (seuil={seuil_opt}) : {cout_opt:,}")
print(f"run_id : {run_id}")

## Comparaison avec le modèle de référence

On compare ici les métriques du modèle LightGBM **sans optimisation** (notebook 3, CV 5-fold)
avec celles du modèle **optimisé par Optuna** (notebook 4, jeu de validation).

> **Note :** Les métriques du baseline sont issues d'une validation croisée 5-fold sur tout le
> dataset d'entraînement. Celles du modèle optimisé sont calculées sur `X_val` (20% non vus
> pendant l'optimisation). Les deux sont comparables mais pas strictement identiques.

In [3]:
# Récupération des métriques du modèle de référence depuis MLflow (notebook 3)
runs_baseline = mlflow.search_runs(
    experiment_names=["credit-scoring"],
    filter_string="tags.mlflow.runName = 'lightgbm'",
    order_by=["start_time DESC"],
    max_results=1,
)
baseline = runs_baseline.iloc[0]

# Tableau de comparaison
df_compare = pd.DataFrame({
    "Modèle":       ["LightGBM baseline (nb. 3)", "LightGBM optimisé (nb. 4)"],
    "AUC":          [baseline["metrics.roc_auc_test_mean"], roc_auc_final],
    "Recall cl. 1": [baseline["metrics.recall_test_mean"],  recall_final],
    "Coût métier":  [baseline["metrics.cout_metier"],       cout_opt],
    "Seuil":        [baseline["metrics.seuil_optimal"],     seuil_opt],
}).set_index("Modèle").round(4)

df_compare

NameError: name 'roc_auc_final' is not defined

In [ ]:
# Graphique comparatif — AUC et Coût métier (les deux métriques clés)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

modeles_labels = ["Baseline", "Optimisé"]
couleurs = ["#95a5a6", "#2ecc71"]

# AUC (↑ meilleur)
axes[0].bar(modeles_labels, df_compare["AUC"], color=couleurs)
axes[0].set_title("AUC (↑ meilleur)")
axes[0].set_ylim(df_compare["AUC"].min() * 0.98, df_compare["AUC"].max() * 1.02)
for i, v in enumerate(df_compare["AUC"]):
    axes[0].text(i, v + 0.0005, f"{v:.4f}", ha="center", va="bottom", fontweight="bold")

# Coût métier (↓ meilleur)
axes[1].bar(modeles_labels, df_compare["Coût métier"], color=couleurs)
axes[1].set_title("Coût métier (↓ meilleur)")
axes[1].set_ylim(0, df_compare["Coût métier"].max() * 1.1)
for i, v in enumerate(df_compare["Coût métier"]):
    axes[1].text(i, v + 200, f"{v:,.0f}", ha="center", va="bottom", fontweight="bold")

fig.suptitle("Baseline vs LightGBM optimisé", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## Feature importance globale

Quelles variables contribuent le plus au modèle, en moyenne sur tout le dataset.

In [ ]:
# Récupération du classifieur et des noms de features après transformation
clf = best_pipeline.named_steps['clf']

ohe_features = (
    best_pipeline.named_steps['preprocessor']
    .named_transformers_['cat']
    .named_steps['ohe']
    .get_feature_names_out(cat_cols)
    .tolist()
)
all_features = num_cols + ohe_features

# Top 20 features par importance
importances = pd.Series(clf.feature_importances_, index=all_features)
top20 = importances.nlargest(20).sort_values().reset_index()
top20.columns = ['feature', 'importance']

fig, ax = plt.subplots(figsize=(10, 8))
create_barh(
    top20, ax,
    x='importance', y='feature',
    sort=False,  # déjà trié par nlargest().sort_values()
    title='Top 20 — Feature importance globale (LightGBM)',
    xlabel='Importance',
)
plt.tight_layout()
plt.savefig('../data/processed/feature_importance_globale.png', dpi=150, bbox_inches='tight')
plt.show()

# Ajout de l'artefact au run MLflow existant
with mlflow.start_run(run_id=run_id):
    mlflow.log_artifact('../data/processed/feature_importance_globale.png')

## Feature importance locale (SHAP)

SHAP permet d'expliquer la décision du modèle **pour un client donné** :
chaque feature reçoit une valeur positive (pousse vers défaut) ou négative (pousse vers remboursement).

In [ ]:
# Transformation des données de validation par le preprocesseur
X_val_transformed = best_pipeline.named_steps['preprocessor'].transform(X_val)

# Sous-échantillon de 500 clients pour accélérer le calcul SHAP
np.random.seed(RAMDOM_STATE)
idx_shap = np.random.choice(len(X_val_transformed), size=500, replace=False)
X_shap   = X_val_transformed[idx_shap]

# Explainer SHAP pour modèles à base d'arbres
explainer  = shap.TreeExplainer(clf)
shap_vals  = explainer.shap_values(X_shap)

# Pour la classification binaire, shap_values peut retourner une liste [classe_0, classe_1]
if isinstance(shap_vals, list):
    shap_vals = shap_vals[1]  # classe positive = défaut de paiement

base_val = explainer.expected_value
if isinstance(base_val, (list, np.ndarray)):
    base_val = base_val[1]

# Graphique beeswarm — distribution des impacts de chaque feature
plt.figure()
shap.summary_plot(
    shap_vals,
    X_shap,
    feature_names=all_features,
    max_display=20,
    show=False,
)
plt.tight_layout()
plt.savefig('../data/processed/shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Probabilités de défaut sur le sous-échantillon SHAP
y_proba_shap = y_pred_proba[idx_shap]

# 3 profils clients représentatifs
idx_haut = y_proba_shap.argmax()                                   # risque élevé
idx_bas  = y_proba_shap.argmin()                                   # faible risque
idx_med  = np.abs(y_proba_shap - np.median(y_proba_shap)).argmin() # cas médian

profils = [
    ('client_risque_eleve',  idx_haut),
    ('client_faible_risque', idx_bas),
    ('client_median',        idx_med),
]

for label, idx in profils:
    shap.waterfall_plot(
        shap.Explanation(
            values=shap_vals[idx],
            base_values=base_val,
            data=X_shap[idx],
            feature_names=all_features,
        ),
        show=False,
    )
    plt.tight_layout()
    plt.savefig(f'../data/processed/shap_{label}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"{label} — proba défaut : {y_proba_shap[idx]:.3f}")

In [13]:
# Logging de tous les artefacts SHAP dans le run MLflow existant
artefacts_shap = [
    '../data/processed/shap_beeswarm.png',
    '../data/processed/shap_client_risque_eleve.png',
    '../data/processed/shap_client_faible_risque.png',
    '../data/processed/shap_client_median.png',
]

with mlflow.start_run(run_id=run_id):
    for path in artefacts_shap:
        mlflow.log_artifact(path)

print("Artefacts SHAP loggés dans MLflow.")

Artefacts SHAP loggés dans MLflow.
